# Chapter 6 · The reviewer

The agent of chapter 5 (contract, feedback and verification gate) with a reviewer: a separate agent, with the same model and read-only tools, that reviews each change the gate has accepted. A free-form reviewer is compared with one that has a rubric asking for the evidence of each criterion of the contract.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 6: the reviewer."""
import ast
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with greedy decoding. Returns the text and the
    input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_TOKENS_STEP,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefill_chunk_size=1024,
                                pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. The rules of chapter 2 and the contract of chapter 4

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name


def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer, require_test=True,
                allows_docs=None):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer. With "require_test" False,
    R7 does not ask for a new check; "allows_docs", if given, decides R3 instead of
    the request."""
    if allows_docs is None:
        allows_docs = bool(ASKS_DOCS.search(task["request"]))
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not allows_docs:
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test or not require_test,
             "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not self.allows_docs():
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists:
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def allows_docs(self):
        """True if the request asks to change the terms and conditions (R3)."""
        return bool(ASKS_DOCS.search(self.task["request"]))

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result


def contract_text(contract):
    """A contract (like data/tasks/Txx/contract.json) written for the agent: goal,
    paths it may change, forbidden paths and numbered acceptance criteria (C1,
    C2…)."""
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    return ("Task contract\n"
            f"Goal: {contract['goal']}\n"
            f"You may change: {', '.join(contract['allowed_paths'])}\n"
            f"Don't touch: {', '.join(contract['forbidden_paths'])}\n"
            f"Acceptance criteria:\n{criteria}\n"
            "The contract overrides AGENTS.md: if it forbids tests/, don't add tests.")


def forbids_tests(contract):
    """True if the contract forbids changing tests/ (for example, a refactoring
    that must pass the tests without touching them)."""
    return rt.path_matches("tests/x.py", contract["forbidden_paths"])


def check_contract(contract, changes, answer):
    """What breaks the contract at the end: files outside the allowed paths or in
    the forbidden ones, and criteria the answer does not explain how it meets."""
    broken = {}
    paths = rt.check_scope(changes["changed"], contract)
    outside = sorted(set(paths["out_of_scope"]) | set(paths["forbidden"]))
    if outside:
        broken["Scope"] = (f"you changed {', '.join(outside)}, which the contract "
                           "does not allow")
    numbers = range(1, len(contract["acceptance_criteria"]) + 1)
    unexplained = [f"C{i}" for i in numbers
                   if not re.search(rf"\bC{i}\b", answer or "")]
    if unexplained:
        broken["Criteria"] = ("your answer does not say how it meets "
                              + ", ".join(unexplained)
                              + " (cite each criterion by its number)")
    return broken


class ToolsWithContract(ToolsWithRules):
    """The tools with the rules of chapter 2 and, on top of them, the contract:
    nothing can be edited or created outside the allowed paths."""

    def __init__(self, tools, task, contract):
        super().__init__(tools, task)
        self.contract = contract

    def allows_docs(self):
        """The contract overrides R3: docs/ can be changed if the contract allows it."""
        return rt.path_matches("docs/terms_and_conditions.md",
                               self.contract["allowed_paths"])

    def broken_rule(self, name, path):
        """The rule or the part of the contract the edit would break, or None."""
        contract = self.contract
        if rt.path_matches(path, contract["forbidden_paths"]):
            return f"Contract: {path} is among the paths you must not touch."
        if not rt.path_matches(path, contract["allowed_paths"]):
            return (f"Contract: {path} is not among the paths you may change ("
                    + ", ".join(contract["allowed_paths"]) + ").")
        return super().broken_rule(name, path)

## 4. Feedback

It can be tried without the model: an edit that makes a test fail, and its summary.

In [ ]:
MAX_FAILURES = 3   # failures summarized for the agent


def summarize_failures(output, folder):
    """The failures of a unittest run, one per test: which test fails, the lines of
    the repository the error goes through (without Python's own) and the last line,
    the one with the error. Paths are left relative to the copy of the repository."""
    failures = []
    prefix = str(folder) + "\\" if "\\" in str(folder) else str(folder) + "/"
    for block in re.split(r"^={50,}$", output, flags=re.M)[1:]:
        header = re.search(r"^(FAIL|ERROR): (\S+)", block, re.M)
        if not header:
            continue
        lines = block.split("-" * 70)[1] if "-" * 70 in block else block
        at = []
        for m in re.finditer(r'File "([^"]+)", line (\d+), in (\S+)\n\s+(.+)', lines):
            if m.group(1).startswith(prefix):
                path = m.group(1)[len(prefix):].replace("\\", "/")
                at.append(f"{path}:{m.group(2)} ({m.group(3)}): {m.group(4).strip()}")
        error = [l for l in lines.strip().splitlines() if l.strip()][-1:]
        failures.append({"test": f"{header.group(1)} {header.group(2)}",
                         "where": at[-3:], "error": error[0].strip() if error else ""})
    return failures[:MAX_FAILURES]


class ToolsWithFeedback(ToolsWithContract):
    """The tools with the contract and, besides, feedback: after each successful
    edit the tests are run and the result of the edit carries a summary of the
    failures. run_tests returns the same summary."""

    def tests(self):
        """The repository's tests, summarized for the agent."""
        result = self.tools.run_tests()
        summary = {"all_pass": result["all_pass"],
                   "result": result["result"]}
        if not result["all_pass"]:
            summary["failures"] = summarize_failures(result["output"],
                                                     self.tools.workspace.folder)
        return summary

    def call(self, name, arguments):
        """Like ToolsWithContract.call, with the tests after each edit."""
        if name == "run_tests":
            result = self.tests()
            self.log.append({"tool": name, "arguments": {}, "ok": True,
                             "seconds": 0, "result": result})
            return result
        result = super().call(name, arguments)
        if name in ("edit", "create_file") and "error" not in result:
            result = dict(result, tests_after_edit=self.tests())
        return result

In [ ]:
task = rt.read_task("T01")
with rt.Workspace() as workspace:
    t = ToolsWithFeedback(rt.Tools(workspace), task, task["contract"])
    t.call("read_file", {"path": "config.py"})
    print(t.call("edit", {"path": "config.py",
                          "old": "FREE_SHIPPING_THRESHOLD = 50",
                          "new": "FREE_SHIPPING_THRESHOLD = 60"}))

## 5. The verification gate

The reference solution of a task passes it; a change that only touches a comment does not.

In [ ]:
def code_without_docstrings(text):
    """The syntax tree of a program, as text, without its docstrings. Comments never
    reach the tree, so two versions that differ only in comments or docstrings give
    the same result. None if it cannot be parsed."""
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return None
    for node in ast.walk(tree):
        body = getattr(node, "body", None)
        if (isinstance(body, list) and body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant)
                and isinstance(body[0].value.value, str)):
            node.body = body[1:] or [ast.Pass()]
    return ast.dump(tree)


def changes_code(workspace, path):
    """True if the file has changed in something other than comments and docstrings."""
    original, new = workspace.source / path, workspace.path(path)
    if not path.endswith(".py") or not original.exists() or not new.exists():
        return True
    before = code_without_docstrings(original.read_text(encoding="utf-8"))
    after = code_without_docstrings(new.read_text(encoding="utf-8"))
    return before is None or after is None or before != after


def verification_gate(workspace, contract):
    """What is missing to accept the task, looking at the repository and not at what
    the agent says: code changes that are not only comments, a new test that fails
    with the original code (if the contract allows touching tests/) and all the
    tests green. Returns a list of sentences; empty if the work can be accepted."""
    changes = rt.differences(workspace.source, workspace.folder)
    lines = diff_lines(changes["diff"])
    missing = []
    code = [r for r in changes["changed"]
            if not r.startswith(("tests/", "docs/", "data/"))]
    if not any(changes_code(workspace, r) for r in code):
        missing.append("the program code has not changed (comments don't count)")
    new_tests = [r for r, l in lines.items() if r.startswith("tests/")
                 and any(re.search(r"def test_|assert", x) for x in l["+"])]
    if not forbids_tests(contract):
        if not new_tests:
            missing.append("there is no new test in tests/")
        else:
            with rt.Workspace() as original:    # the new tests, with the old code
                for path in new_tests:
                    destination = original.path(path)
                    destination.write_text(workspace.path(path).read_text("utf-8"),
                                           encoding="utf-8")
                before = rt.Tools(original).run_tests()
            if before["all_pass"]:
                missing.append("your new tests also pass with the original code, so "
                               "they don't check the change the request asks for")
    now = rt.Tools(workspace).run_tests()
    if not now["all_pass"]:
        failures = summarize_failures(now["output"], workspace.folder)
        missing.append("the tests don't pass: "
                       + "; ".join(f"{f['test']} → {f['error']}" for f in failures))
    return missing

In [ ]:
with rt.Workspace() as workspace:
    rt.apply_solution(rt.Tools(workspace), task["folder"] / "solution")
    print("Solution:", verification_gate(workspace, task["contract"]))
with rt.Workspace() as workspace:
    t = rt.Tools(workspace)
    t.call("edit", {"path": "shipping.py", "old": "Shipping cost of an order",
                    "new": "Shipping cost (free only in the contiguous US) "
                           "of an order"})
    print("Comment:", verification_gate(workspace, task["contract"]))

## 6. The reviewer

Its tools only read. It can be checked without the model.

In [ ]:
MAX_REVIEWER_STEPS = 10  # model calls in each review


MAX_REVIEWS = 2          # times the reviewer can send the work back


ESCALATED = ("NOT DONE: the reviewer has rejected the change twice; it is waiting "
             "for a person to review it.")


READ_ONLY = ("list_files", "read_file", "search", "run_tests")


REVIEWER_SYSTEM = (
    "You are the change reviewer of the North Cycles repository, a bike store in "
    "Portland, Oregon. Another agent has made a change to solve a request and says "
    "it is done. Your job is to check whether the change really solves the "
    "request. You can list, read and search files and run the tests, but you "
    "cannot change anything. Use one tool at a time. When you are done, answer "
    "without calling any tool, with only a JSON.")


ASK_FREE = """Review this change.

Request:
{request}

Change (diff):
{diff}

What the author says:
{answer}

At the end, answer with this JSON:
{{"approved": true or false, "comments": ["…"]}}"""


ASK_RUBRIC = """Review this change criterion by criterion.

Request:
{request}

Acceptance criteria:
{criteria}

Change (diff):
{diff}

What the author says:
{answer}

For each criterion, look in the code (not in what the author says) to see whether
it is met, and give the evidence: file and line, or the test that checks it. Also
check whether the change touches anything the request does not ask for. At the end,
answer with this JSON:
{{"criteria": [{{"id": "C1", "met": true or false, "evidence": "…"}}],
  "beyond_request": ["…"], "approved": true or false}}
Approve only if every criterion is met."""


class ReadOnlyTools:
    """The reviewer's tools: the usual ones, but only those that do not change
    anything. Any other one is rejected."""

    def __init__(self, tools):
        self.tools = tools
        self.log = []

    def call(self, name, arguments):
        """Like Tools.call, only for the reading tools."""
        if name not in READ_ONLY:
            result = {"error": f'The reviewer cannot use "{name}": only '
                               + ", ".join(READ_ONLY) + "."}
        else:
            result = self.tools.call(name, arguments)
        self.log.append({"tool": name, "ok": "error" not in result})
        return result


def read_verdict(answer):
    """The last JSON object of the reviewer's answer, or None if there is none
    with the key "approved"."""
    for piece in reversed(re.findall(r"\{.*\}", answer, re.S)):
        try:
            verdict = json.loads(piece, strict=False)
        except ValueError:
            continue
        if isinstance(verdict, dict) and "approved" in verdict:
            return verdict
    return None


def review(tok, model, task, contract, workspace, author_answer, rubric, verbose=False):
    """One review: a separate agent, with its own conversation and read-only tools
    on the copy of the repository, that returns its verdict. Without a readable
    verdict, the change is approved (the reviewer does not block for failing)."""
    changes = rt.differences(workspace.source, workspace.folder)
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    request_template = ASK_RUBRIC if rubric else ASK_FREE
    messages = [{"role": "system", "content": REVIEWER_SYSTEM},
                {"role": "user", "content": request_template.format(
                    request=task["request"], criteria=criteria,
                    diff=changes["diff"][:3000], answer=author_answer[:800])}]
    tools = ReadOnlyTools(rt.Tools(workspace))
    verdict, steps = None, 0
    for steps in range(1, MAX_REVIEWER_STEPS + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, _, _ = generate(tok, model, text)
        calls = read_calls(answer)
        if not calls:
            verdict = read_verdict(answer)
            break
        messages.append({"role": "assistant", "content": answer.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            result = (tools.call(call["name"], call["arguments"])
                      if call["name"] else {"error": "Invalid call."})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    if verbose:
        print(f"    REVIEWER ({steps} steps): "
              f"{json.dumps(verdict, ensure_ascii=False)[:300]}")
    return {"approved": True if verdict is None else bool(verdict.get("approved")),
            "verdict": verdict, "steps": steps,
            "calls": len(tools.log)}


def comments_for_author(verdict):
    """What the reviewer objects to, in sentences for the author."""
    if not verdict:
        return "the reviewer does not approve the change"
    sentences = [f"{c.get('id', '?')} is not met: {c.get('evidence', '')}"
                 for c in verdict.get("criteria", []) if not c.get("met", True)]
    sentences += [f"beyond the request: {x}" for x in verdict.get("beyond_request", [])]
    sentences += [str(x) for x in verdict.get("comments", [])]
    return "; ".join(sentences) or "the reviewer does not approve the change"

In [ ]:
with rt.Workspace() as workspace:
    t = ReadOnlyTools(rt.Tools(workspace))
    print(t.call("edit", {"path": "config.py", "old": "a", "new": "b"}))

## 7. The agent with the reviewer

The new free-shipping thresholds (T15), with the free-form reviewer and with the rubric.

In [ ]:
def agent(tok, model, task, reviewer=None, max_steps=MAX_STEPS, verbose=False):
    """The agent of chapter 5 (contract, feedback and verification gate) with an
    optional reviewer: None, "free" or "rubric". When the gate accepts a "done",
    the reviewer reviews it; if it does not approve it, its comments go back to the
    author. After MAX_REVIEWS rejections, the task waits for a person.
    Returns the row of results."""
    start = time.time()
    contract = task["contract"]
    request = task["request"] + "\n\n" + contract_text(contract)
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": request}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    reviews = []
    with rt.Workspace() as workspace:
        base = rt.Tools(workspace)
        tools = ToolsWithFeedback(base, task, contract)
        for steps in range(1, max_steps + 1):
            text, trimmed = trim_conversation(tok, messages)
            if trimmed is None:
                break
            answer, n_input, n_output = generate(tok, model, text)
            n_in, n_out = n_in + n_input, n_out + n_output
            calls = read_calls(answer)
            if not calls:
                messages.append({"role": "assistant", "content": answer})
                if verbose:
                    print(f"[{steps}] {answer}")
                if says_done(answer) and rejections < MAX_REJECTIONS:
                    changes = rt.differences(workspace.source, workspace.folder)
                    broken = check_rules(task, base.log, changes, answer,
                                         require_test=not forbids_tests(contract),
                                         allows_docs=tools.allows_docs())
                    broken.update(check_contract(contract, changes, answer))
                    broken.pop("R7", None)
                    missing = verification_gate(workspace, contract)
                    if missing:
                        broken["Verification"] = "; ".join(missing)
                    if broken:
                        rejections += 1
                        notice = ("It isn't done yet: "
                                  + "; ".join(f"{r}: {t}"
                                              for r, t in sorted(broken.items()))
                                  + ". Keep working.")
                        messages.append({"role": "user", "content": notice})
                        if verbose:
                            print(f"    GATE: {notice}")
                        continue
                if reviewer and says_done(answer):
                    if len(reviews) == MAX_REVIEWS:   # it has already rejected it twice
                        final = ESCALATED
                        break
                    this_review = review(tok, model, task, contract, workspace,
                                         answer, rubric=reviewer == "rubric",
                                         verbose=verbose)
                    reviews.append(this_review)
                    if not this_review["approved"]:
                        messages.append({"role": "user", "content":
                                         "The reviewer does not approve the change: "
                                         + comments_for_author(this_review["verdict"])
                                         + ". Fix it and answer DONE again."})
                        continue
                final = answer
                break
            thought = answer.split("<tool_call>")[0].strip()
            messages.append({"role": "assistant", "content": thought,
                             "tool_calls": [{"type": "function", "function": c}
                                            for c in calls if c["name"]]})
            for call in calls:
                if call["name"] is None:
                    result = {"error": "The tool call cannot be understood."}
                else:
                    result = tools.call(call["name"], call["arguments"])
                if verbose:
                    print(f"[{steps}] {call['name']}"
                          f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
                messages.append({"role": "tool", "name": call["name"] or "?",
                                 "content": json.dumps(result, ensure_ascii=False)})
        evaluation = rt.evaluate_attempt(workspace, task)
    done = final is not None and says_done(final)
    return {"task": task["id"], "type": task["meta"]["type"],
            "solved": evaluation["solved"], "says_done": done,
            "false_done": done and not evaluation["solved"],
            "visible": evaluation["visible"], "hidden": evaluation["hidden"],
            "out_of_scope": evaluation["out_of_scope"], "rejections": rejections,
            "reviews": reviews,
            "steps": steps, "input_tokens": n_in, "output_tokens": n_out,
            "seconds": round(time.time() - start, 1),
            "answer": final or "(no final answer)",
            "calls": [[r["tool"], r["arguments"].get("path", ""), r["ok"]]
                      for r in base.log],
            "diff": evaluation["diff"]}


VARIANTS = {
    "no reviewer": {},
    "free-form reviewer": {"reviewer": "free"},
    "reviewer with rubric": {"reviewer": "rubric"},
}

In [ ]:
for name in ("free-form reviewer", "reviewer with rubric"):
    print(f"--- {name}")
    row = agent(tok, model, rt.read_task("T15"), verbose=True, **VARIANTS[name])
    print("Solved:", row["solved"], "· Answer:", row["answer"][:120])

## 8. The twenty-four tasks with the three variants

By default a sample is run; with `COMPLETE = True`, the book's whole experiment.

In [ ]:
MEDIUM_TASKS = [f"T{n}" for n in range(21, 31)]   # the extended bank of chapter 5


BANK = TASKS + MEDIUM_TASKS


def summary(rows):
    """The totals of a variant, with what the reviewer did."""
    reviews = [r for f in rows for r in f["reviews"]]
    return {"tasks": len(rows),
            "solved": sum(r["solved"] for r in rows),
            "says_done": sum(r["says_done"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "tests_red": sum(not r["visible"] for r in rows),
            "reviews": len(reviews),
            "reviewer_rejections": sum(not r["approved"] for r in reviews),
            "no_verdict": sum(r["verdict"] is None for r in reviews),
            "steps": sum(r["steps"] for r in rows),
            "reviewer_steps": sum(r["steps"] for r in reviews),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, options, tasks=BANK):
    """One variant with every task: its rows and its summary."""
    rows = []
    for task_id in tasks:
        row = agent(tok, model, rt.read_task(task_id), **options)
        rows.append(row)
        print(f"  {task_id} {'solved' if row['solved'] else 'no    '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  reviews={len(row['reviews'])}"
              f" ({sum(not r['approved'] for r in row['reviews'])} rejections)"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False two tasks are run (about 35 minutes on Colab's T4);
# with True, the chapter's whole experiment (about twenty hours on the T4).
COMPLETE = False
tasks = BANK if COMPLETE else ["T19", "T21"]
results = {}
for name, options in VARIANTS.items():
    print(f"== {name}")
    results[name] = run_all(tok, model, options, tasks=tasks)
    print(results[name]["summary"])

The book's full results, with all the tasks, are saved in `data/results/chapter_6/`, which `prepare` has downloaded:

In [ ]:
import json
saved = json.loads((ROOT / "data" / "results" / "chapter_6" / "results.json").read_text(encoding="utf-8"))
for name, r in saved.items():
    print(name, r["summary"])

## Exercises

The results files the exercises cite are in `data/results/chapter_6/`, which `prepare` downloads with the data.

1. What is the difference between the verification gate of chapter 5 and the reviewer? Give an example of something only each one can see. *(No GPU.)*

2. Read the verdicts of the reviewer with a rubric on the tasks it approves. Is all the evidence correct? Check at least two by opening the file and the line they cite. *(No GPU.)*

3. Remove from the rubric message the phrase "look in the code (not in what the author says)". Do the verdicts change? Try it with the tasks in which the author says it is done. The results with the four tasks in which the reviewer's verdict decides something (T03, T11, T15 and T21) are already in `results_exercises.json`; to see it for yourself, the free-shipping thresholds one (T15) is enough. *(No GPU with the saved results; GPU, about 25 minutes, with that task.)*

4. The reviewer only reviews what gets past the gate. Make it review also when the author ends without saying "done" and check whether any of those tasks was, in fact, correctly solved. The three tasks that end correctly solved without a "done" with the rubric reviewer (T05, T21 and T25) are in `results.json`. *(No GPU with the saved results; GPU, about 20 minutes, with T05 and T21.)*

5. Write a rubric to review your own team's changes, with no more than six points, each one checkable with a concrete piece of evidence. *(No GPU.)*

6. What would happen if the reviewer could edit? Think about the magic numbers task: who would answer for the changes? *(No GPU.)*

7. The reviewer and the author are the same model. Find a task in which the reviewer repeats the author's mistake. What would it need to have in front of it so as not to repeat it? *(No GPU.)*

8. Change `MAX_REVIEWS` to 1 and to 3. How many tasks end up escalated? Does any improve with one more round? The results with the four tasks in which the reviewer's verdict decides something (T03, T11, T15 and T21) are already in `results_exercises.json`; to see it for yourself, the free-shipping thresholds one (T15) is enough. *(No GPU with the saved results; GPU, about 50 minutes, with that task and both limits.)*

9. Work out the cost of the reviewer with a rubric: model calls and tokens of the reviews, compared with the author's. How much would it cost to review all the tasks, and not only the ones that pass the gate? *(No GPU.)*

10. **Project.** Put a read-only reviewer with a rubric in front of a coding agent you use, commercial or the book's, for a week. Write down how many changes it rejects, how many of those rejections were fair, and how much time it saves you or costs you.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Google Engineering Practices: Code Review](https://google.github.io/eng-practices/review/). [Judging LLM-as-a-Judge](https://arxiv.org/abs/2306.05685) (Zheng et al., 2023). The North Cycles store, its program and its tasks are fictional. Original code and experiments.